# Guided Lab：完成訂單查詢的工具呼叫流程

本實驗依據 [工具呼叫講義](tool_calling.md)，使用講義的訂單資料與 `gpt-5-nano`。你將補完客服程式，觀察模型的工具請求、應用程式的執行結果，以及最後的客服回答。

**時間：25 分鐘。** 

## 學習目標

完成後，你應能：

1. 閱讀 `@tool`、型別註記及 docstring，並用 `bind_tools()` 提供工具。
2. 補完工具執行流程，確認請求 `id` 與結果 `tool_call_id` 的對應。
3. 測試系統提示詞如何處理缺少編號與查無訂單。
4. 使用 `create_agent()`，從回傳的 `messages` 觀察自動執行流程。

## 課前準備

先完成 [Notebook 與 API 環境設定](../setup/first_ipynb_openai_api.md)，確認所選 Kernel 已安裝 `langchain`、`langchain-openai`，且可使用課程模型。安裝與帳號設定不計入 25 分鐘。




In [ ]:
# load API key from environment variable 
from dotenv import load_dotenv

load_dotenv()  # take environment variables from .env.

# 如果 .env 不在目前專案目錄下:
# env_path = 'your_path_to_env_file/.env'
# load_dotenv(dotenv_path=env_path)  


## 25 分鐘流程

| 時間 | 操作 | 完成證據 |
| --- | --- | --- |
| 0～4 分鐘 | 閱讀工具並直接查詢資料 | 看見 A1024 的字典結果 |
| 4～8 分鐘 | TODO 1：補提示詞並綁定工具 | 三項處理規則及已綁定的模型 |
| 8～15 分鐘 | TODO 2：補完手動工具呼叫流程 | 請求、結果、相同的識別碼及回答 |
| 15～19 分鐘 | 執行兩個特殊情境 | 完成三列觀察表 |
| 19～23 分鐘 | TODO 3：改用 Agent 查詢 | Agent 的訊息歷程與回答 |
| 23～25 分鐘 | 回答兩題並繳交 | 已儲存輸出與簡答的 Notebook |

## 步驟 1：閱讀工具並查詢資料（4 分鐘）

工具接收訂單編號 `order_id`，從字典查詢資料，回傳狀態與預計送達日。這裡保留講義的歷史日期作為固定模擬資料。

`@tool` 將函數包裝成工具物件；型別註記描述輸入與輸出，docstring 說明用途。先執行下列程式，直接把參數字典交給工具，確認資料內容。


In [ ]:
from langchain.tools import tool

@tool
def get_order_status(order_id: str) -> dict:
    """依客戶提供的訂單編號，查詢訂單狀態與預計送達日。

    Args:
        order_id: 客戶提供的訂單編號。
    """
    orders = {
        "A1024": {"status": "Shipped", "estimated_delivery": "2023-07-25"},
        "B2048": {"status": "Processing", "estimated_delivery": "2023-07-30"},
        "C4096": {"status": "Delivered", "estimated_delivery": "2023-07-20"},
    }
    return orders.get(
        order_id,
        {"status": "Order ID not found", "estimated_delivery": None},
    )

print(get_order_status.name)
print(get_order_status.description)
print(get_order_status.invoke({"order_id": "A1024"}))


**檢查點：** 查詢結果應包含 `Shipped` 與 `2023-07-25`。這一步是你的程式直接執行工具，尚未呼叫模型。

## 步驟 2：補上工具使用規則（4 分鐘）

系統提示詞決定工具的使用時機與回答依據。`bind_tools()` 則把可用工具提供給模型。

**TODO 1：** 將提示詞中的三個 TODO 改成完整規則，並補完綁定工具的一行。不要把 A1024 的答案直接寫入提示詞。


In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

system_prompt = """
你是電商客服助理，請以繁體中文簡短回答。
客戶詢問訂單狀態且提供編號時，使用 get_order_status。
將客戶提供的訂單編號原樣傳入 order_id。
與訂單狀態無關的問題不使用工具。

TODO：缺少訂單編號時，應如何處理？
TODO：工具回傳 Order ID not found 時，應如何處理？
TODO：回答中的訂單狀態與日期，應以什麼為依據？
"""

chat_model = init_chat_model("gpt-5-nano")
tools = [get_order_status]
model_with_tools = ...  # TODO 1：呼叫 chat_model 的 bind_tools()
tools_by_name = {t.name: t for t in tools}


**提示：** 對照講義「設計系統提示詞」一節，分別寫出先詢問、請確認編號與依據工具結果的規則。

## 步驟 3：補完手動工具呼叫流程（7 分鐘）

下列函數接收一則客戶問題，建立本次查詢的訊息列表。模型先回傳 `AIMessage`；若內含 `tool_calls`，由程式執行工具，再把結果送回模型。

每筆請求的 `name` 指定工具、`args` 提供參數、`id` 識別請求，`type` 為 `tool_call`。這次把**完整請求**傳給工具的 `invoke()`，回傳的 `ToolMessage` 會帶有對應的 `tool_call_id`。

**TODO 2：** 補完四行 `...`。本函數示範講義的一次 Model → Tool → Model 流程；若模型還要求工具，程式會提示你保留紀錄，在步驟 5 用 Agent 觀察後續流程。


In [ ]:
def run_manual(user_input):
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_input),
    ]
    ai_message = model_with_tools.invoke(messages)
    print("工具請求：", ai_message.tool_calls)
    messages.append(...)  # TODO 2a：保留模型回傳的請求訊息

    if ai_message.tool_calls:
        for tool_call in ai_message.tool_calls:
            selected_tool = tools_by_name[tool_call["name"]]
            tool_message = ...  # TODO 2b：傳入完整 tool_call 執行工具
            print("工具結果：", tool_message.content)
            print("ID 對應：", tool_call["id"], tool_message.tool_call_id)
            assert tool_message.tool_call_id == tool_call["id"]
            messages.append(...)  # TODO 2c：加入工具結果訊息

        final_response = ...  # TODO 2d：以更新後的 messages 再呼叫模型
        messages.append(final_response)
    else:
        final_response = ai_message

    if final_response.tool_calls:
        print("模型仍提出工具請求；保留此紀錄，於步驟 5 觀察 Agent。")
    else:
        print("客服回答：", final_response.content)
    print("訊息順序：", [type(m).__name__ for m in messages])
    return messages

normal_messages = run_manual("請查詢訂單 A1024 的狀態與預計送達日。")


**檢查點：**

- 請求應指定 `get_order_status`，參數為 `{"order_id": "A1024"}`。
- 工具結果應包含 `Shipped` 與 `2023-07-25`；客服回答的意思應與資料一致。
- 印出的兩個 ID 應相同。從使用者訊息起，順序應為 `HumanMessage → AIMessage → ToolMessage → AIMessage`，前面另有 `SystemMessage`。

**停下來想一想：** 只看見「工具請求」時，能知道 Python 工具已經執行了嗎？哪一行才真正執行工具？

## 步驟 4：測試缺少編號與查無訂單（4 分鐘）

沿用同一套提示詞與函數，執行下列兩個獨立問題。每次呼叫都建立新的訊息列表，避免前一次的 A1024 影響缺少編號的測試。


In [ ]:
missing_messages = run_manual("我的訂單什麼時候會到？")
unknown_messages = run_manual("請查詢訂單 Z9999 的狀態。")


將實際結果填入表格。自然語言用字可以不同，請檢查工具呼叫與回答是否符合規則。

| 輸入 | 預期行為 | 實際工具名稱與 args（沒有則填「無」） | 實際回答摘要／是否符合 |
| --- | --- | --- | --- |
| A1024 | 查詢工具，依結果回答 |  |  |
| 缺少編號 | 先詢問編號 |  |  |
| Z9999 | 查詢工具，告知查無資料並請確認 |  |  |

**若不符合：** 對照 TODO 1 的規則，修改相關一句並重跑該案例。若修改了提示詞，記得先重跑步驟 2 的儲存格。

## 步驟 5：用 Agent 完成同一個查詢（4 分鐘）

`create_agent()` 建立的 Agent 會管理工具執行與訊息更新。輸入是含 `messages` 的 State 字典；回傳的 State 保留這次查詢的訊息歷程。

**TODO 3：** 在兩處 `...` 填入訊息列表的鍵名。執行後，比較手動版與 Agent 版的工具參數、工具結果及最後回答。


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt=system_prompt,
)

response = agent.invoke({
    ...: [HumanMessage(content="請查詢訂單 A1024 的狀態與預計送達日。")]
})
agent_messages = response[...]
for message in agent_messages:
    print(type(message).__name__, message.content)
    if getattr(message, "tool_calls", None):
        print("工具請求：", message.tool_calls)
    if hasattr(message, "tool_call_id"):
        print("結果對應 ID：", message.tool_call_id)
print("最終回答：", agent_messages[-1].content)


**檢查點：** 在訊息歷程找到工具請求及其 `ToolMessage`，確認 ID 相符。兩個版本的回答文字可以不同，但訂單資料應一致。

## 步驟 6：簡答與繳交（2 分鐘）

1. 手動版的哪一行真正執行工具？為什麼要同時保留工具請求的 `AIMessage` 與結果的 `ToolMessage`？
2. 改用 Agent 後，Model 與 Harness 各負責什麼？列出兩項由 Harness 接手的手動工作。

**繳交：** 一份完成 TODO 的 Notebook，保留三個案例及 Agent 的執行輸出，並填寫觀察表與兩題簡答。

## 參考資料

- [課程講義：工具呼叫](tool_calling.md)
- [LangChain：工具定義](https://docs.langchain.com/oss/python/langchain/tools)
- [LangChain：工具執行迴圈](https://docs.langchain.com/oss/python/langchain/models#tool-execution-loop)
- [LangChain：Agent](https://docs.langchain.com/oss/python/langchain/agents)
